In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Kunci latihan dan ujian tingkat intermediate

Jalankan dari atas ke bawah.


In [2]:
# Pustaka adalah kumpulan fungsi siap pakai, bukan rak buku.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


In [3]:
# Satu ide: precision latihan dari rumus.
latihan_precision = 6 / (6 + 3)
print("latihan_precision", latihan_precision)
assert latihan_precision == 6 / 9


latihan_precision 0.6666666666666666


In [4]:
# Satu ide: recall latihan dari rumus.
latihan_recall = 6 / (6 + 4)
print("latihan_recall", latihan_recall)
assert latihan_recall == 6 / 10


latihan_recall 0.6


In [5]:
# Satu ide: bagian yang cocok bila tebakan selalu tidak hujan.
y_jarang = np.array([0] * 90 + [1] * 10)
tebak_jarang = np.zeros(100, dtype=int)
latihan_cocok = float((tebak_jarang == y_jarang).mean())
print("latihan_cocok", latihan_cocok)
assert latihan_cocok == 90 / 100


latihan_cocok 0.9


In [6]:
# Satu ide: recall latihan pada tebakan yang sama.
penemu = int(((tebak_jarang == 1) & (y_jarang == 1)).sum())
hujan = int((y_jarang == 1).sum())
print("latihan_recall_jarang", float(penemu / hujan))
assert hujan == 10 and penemu == 0


latihan_recall_jarang 0.0


## Ujian tingkat intermediate


In [7]:
# Satu ide: data ujian sintetis. Kelas hujan lebih sedikit.
# Menurut dokumentasi scikit-learn, make_classification mengundi soal klasifikasi.
# n_samples adalah banyaknya baris.
# n_features adalah banyaknya kolom semuanya: kolom informatif, kolom redundan, kolom salinan, dan sisanya kolom acak yang tidak berguna.
# n_informative adalah banyaknya kolom informatif. Tiap kelas punya kelompok sebaran normal (simpangan baku 1) di sekitar sudut hypercube berdimensi sebanyak itu.
# Hypercube adalah kotak bersudut di beberapa dimensi. Bukan istilah baru.
# Di dalam tiap kelompok, kolom informatif diundi dari sebaran normal baku (rata-rata 0, simpangan baku 1), lalu digabung secara linear, sebelum kelompok diletakkan di sudut hypercube.
# n_redundant adalah banyaknya kolom kombinasi linear acak dari kolom informatif.
# n_repeated adalah banyaknya kolom salinan, diambil acak dari kolom informatif dan redundan.
# n_clusters_per_class adalah banyaknya kelompok pada tiap kelas.
# class_sep mengalikan ukuran hypercube. Panjang sisinya 2 kali class_sep. Makin besar, kelompok makin berjauhan dan soal makin mudah.
# flip_y adalah bagian baris yang kelasnya diundi ulang secara acak.
# weights adalah proporsi baris yang ditugaskan ke tiap kelas. Proporsi akhir tidak persis sama dengan weights bila flip_y bukan nol.
# random_state membuat undian pembuatan data ini berulang selama angkanya sama.
# Nama parameter ini bukan istilah baru.
X_ujian, y_ujian = make_classification(
    n_samples=400,
    n_features=20,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    weights=[0.85, 0.15],
    class_sep=1.3,
    flip_y=0.02,
    random_state=SEED,
)
print("ujian_baris", int(X_ujian.shape[0]))
print("ujian_bagian_hujan", float(y_ujian.mean()))


ujian_baris 400
ujian_bagian_hujan 0.16


In [8]:
# Satu ide: sisihkan data uji. stratify menjaga proporsi kelas. Bukan istilah baru.
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_ujian,
    y_ujian,
    test_size=0.3,
    random_state=SEED,
    shuffle=True,
    stratify=y_ujian,
)
print("ujian_n_latih", int(len(y_latih)))
print("ujian_n_uji", int(len(y_uji)))


ujian_n_latih 280
ujian_n_uji 120


In [9]:
# Satu ide: baseline selalu menebak kelas yang paling banyak.
kelas_banyak = int(np.bincount(y_latih).argmax())
tebakan_baseline = np.full_like(y_uji, kelas_banyak)
print("ujian_cocok_baseline", float((tebakan_baseline == y_uji).mean()))
assert np.array_equal(
    tebakan_baseline, np.full_like(y_uji, int(np.bincount(y_latih).argmax()))
)


ujian_cocok_baseline 0.8416666666666667


In [10]:
# Satu ide: recall baseline.
print(
    "ujian_recall_baseline",
    float(recall_score(y_uji, tebakan_baseline, pos_label=1, zero_division=0)),
)


ujian_recall_baseline 0.0


In [11]:
# Satu ide: latih pohon pendek dan cetak skor data latih.
pohon = DecisionTreeClassifier(max_depth=2, random_state=SEED)
pohon.fit(X_latih, y_latih)
skor_latih_dangkal = float(pohon.score(X_latih, y_latih))
print("ujian_skor_latih_dangkal", skor_latih_dangkal)


ujian_skor_latih_dangkal 0.9642857142857143


In [12]:
# Satu ide: skor pohon pendek pada data uji.
tebakan_uji = pohon.predict(X_uji)
skor_uji_dangkal = float((tebakan_uji == y_uji).mean())
print("ujian_skor_uji_dangkal", skor_uji_dangkal)


ujian_skor_uji_dangkal 0.9166666666666666


In [13]:
# Satu ide: empat kotak pohon pendek pada data uji.
tn, fp, fn, tp = confusion_matrix(y_uji, tebakan_uji, labels=[0, 1]).ravel()
print("ujian_tepat_negatif", int(tn))
print("ujian_salah_positif", int(fp))
print("ujian_salah_negatif", int(fn))
print("ujian_tepat_positif", int(tp))


ujian_tepat_negatif 98
ujian_salah_positif 3
ujian_salah_negatif 7
ujian_tepat_positif 12


In [14]:
# Satu ide: precision pohon pendek pada data uji.
precision_dangkal = float(
    precision_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)
)
print("ujian_precision", precision_dangkal)
assert precision_dangkal == tp / (tp + fp)


ujian_precision 0.8


In [15]:
# Satu ide: recall pohon pendek pada data uji.
recall_dangkal = float(
    recall_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)
)
print("ujian_recall", recall_dangkal)
assert recall_dangkal == tp / (tp + fn)


ujian_recall 0.631578947368421


In [16]:
# Satu ide: skor pohon tanpa batas pada data latih.
pohon_dalam = DecisionTreeClassifier(max_depth=None, random_state=SEED)
pohon_dalam.fit(X_latih, y_latih)
skor_latih_dalam = float(pohon_dalam.score(X_latih, y_latih))
print("ujian_skor_latih_dalam", skor_latih_dalam)


ujian_skor_latih_dalam 1.0


In [17]:
# Satu ide: skor pohon tanpa batas pada data uji.
skor_uji_dalam = float(pohon_dalam.score(X_uji, y_uji))
print("ujian_skor_uji_dalam", skor_uji_dalam)


ujian_skor_uji_dalam 0.8916666666666667


In [18]:
# Satu ide: simpulan ujian.
# Melewatkan hujan lebih merugikan, jadi metrik yang diutamakan adalah recall.
# Model yang dilaporkan adalah yang selisih skornya lebih kecil, selama recall-nya
# mengalahkan baseline.
selisih_dalam = float(skor_latih_dalam - skor_uji_dalam)
selisih_dangkal = float(skor_latih_dangkal - skor_uji_dangkal)
print("ujian_selisih_dalam", selisih_dalam)
print("ujian_selisih_dangkal", selisih_dangkal)
if selisih_dalam > selisih_dangkal:
    menghafal = "dalam"
else:
    menghafal = "dangkal"
print("ujian_menghafal", menghafal)
recall_baseline = float(
    recall_score(y_uji, tebakan_baseline, pos_label=1, zero_division=0)
)
if menghafal == "dalam" and recall_dangkal > recall_baseline:
    print("ujian_model_dipilih", "dangkal")
else:
    print("ujian_model_dipilih", "periksa_lagi")
print("ujian_metrik_dipilih", "recall")
print("ujian_nama_kebocoran", "data leakage")
print("ujian_rata_boleh", "data latih")
assert menghafal == "dalam"
assert recall_dangkal > recall_baseline


ujian_selisih_dalam 0.10833333333333328
ujian_selisih_dangkal 0.04761904761904767
ujian_menghafal dalam
ujian_model_dipilih dangkal
ujian_metrik_dipilih recall
ujian_nama_kebocoran data leakage
ujian_rata_boleh data latih
